# Smart Grid Energy Forecasting

This notebook trains two XGBoost models and predicts daily electricity load from a sample input.

- **Lag model:** uses the realized load from 1 and 7 days earlier.
- **Fallback model:** works without earlier load values.

## 1. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

## 2. Load the daily data

In [ ]:
# Read the prepared daily dataset.
data_path = Path('combined.csv')
data = pd.read_csv(data_path, parse_dates=['date'])

# Convert every date to the same UTC format.
data['date'] = pd.to_datetime(data['date'], utc=True)

print(f'Rows: {len(data):,}')
data.head()

## 3. Create model features

In [ ]:
def make_features(frame, include_lags=False):
    """Create generation, weather, calendar, and optional lag features."""
    features = pd.DataFrame({
        'current_energy_generation': frame['current_energy_generation'],
        'temperature_celsius': frame['temperature_celsius'],
        'day_of_week': frame['date'].dt.dayofweek,
        # Sine and cosine describe the yearly seasonal cycle.
        'day_of_year_sin': np.sin(2 * np.pi * frame['date'].dt.dayofyear / 365.25),
        'day_of_year_cos': np.cos(2 * np.pi * frame['date'].dt.dayofyear / 365.25),
    })

    if include_lags:
        # These are actual loads from 1 day and 7 days earlier.
        features['load_lag_1'] = frame['realized_load'].shift(1)
        features['load_lag_7'] = frame['realized_load'].shift(7)

    return features

## 4. Train and evaluate the models

The first 80% of the dates are used for training and the final 20% for testing. Keeping the split chronological gives a realistic evaluation on future dates.

In [ ]:
def new_model():
    """Return a small XGBoost regression model."""
    return XGBRegressor(
        n_estimators=600,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.90,
        objective='reg:squarederror',
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
    )


def train_and_score(features, target):
    """Train chronologically and return the model and its test scores."""
    split = int(len(features) * 0.80)
    x_train, x_test = features.iloc[:split], features.iloc[split:]
    y_train, y_test = target.iloc[:split], target.iloc[split:]

    model = new_model()
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)

    scores = {
        'R²': r2_score(y_test, predictions),
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': mean_squared_error(y_test, predictions) ** 0.5,
    }
    return model, scores

In [ ]:
target = data['realized_load']

# Train the fallback model with generation, temperature, and date features.
fallback_features = make_features(data)
fallback_model, fallback_scores = train_and_score(fallback_features, target)

# The first seven rows have no complete lag history, so remove them.
lag_features = make_features(data, include_lags=True).dropna()
lag_target = target.loc[lag_features.index]
lag_model, lag_scores = train_and_score(lag_features, lag_target)

# Compare both models.
pd.DataFrame({
    'Lag model': lag_scores,
    'Fallback model': fallback_scores,
}).T.round(3)

## 5. Predict from a sample input

Change the values in `sample` to make another prediction. The lag values must represent the realized loads from one and seven days before the prediction date.

In [ ]:
# Example values for 15 January 2024.
sample = {
    'date': '2024-01-15',
    'generation': 15774.93,
    'temperature': 0.23,
    'load_yesterday': 13605.83,
    'load_7_days_ago': 15682.01,
}

# Convert the date into the same features used during training.
date = pd.Timestamp(sample['date'])
sample_features = pd.DataFrame([{
    'current_energy_generation': sample['generation'],
    'temperature_celsius': sample['temperature'],
    'day_of_week': date.dayofweek,
    'day_of_year_sin': np.sin(2 * np.pi * date.dayofyear / 365.25),
    'day_of_year_cos': np.cos(2 * np.pi * date.dayofyear / 365.25),
    'load_lag_1': sample['load_yesterday'],
    'load_lag_7': sample['load_7_days_ago'],
}])

predicted_load = float(lag_model.predict(sample_features)[0])
difference = sample['generation'] - predicted_load
difference_percent = difference / predicted_load * 100

# A difference within 5% is treated as balanced.
if abs(difference_percent) <= 5:
    status = 'Balanced'
elif difference > 0:
    status = 'Overproducing'
else:
    status = 'Underproducing'

result = pd.DataFrame([{
    'date': sample['date'],
    'generation': sample['generation'],
    'predicted_load': predicted_load,
    'difference': difference,
    'difference_percent': difference_percent,
    'status': status,
}])

result.round(2)

## 6. Predict without lag values

Use the fallback model if the previous loads are unknown.

In [ ]:
# The fallback model uses only the first five features.
fallback_sample = sample_features[fallback_features.columns]
fallback_prediction = float(fallback_model.predict(fallback_sample)[0])

print(f'Fallback predicted load: {fallback_prediction:,.2f}')